# CODEFEST — Generar índice FAISS con BAAI/bge-m3 (plan B, Google Colab)

Notebook autocontenido: no necesita clonar el repo, solo el archivo
`chunks.jsonl` (163,625 chunks del corpus, generado por el pipeline de
chunking) subido a Google Drive.

**Antes de correr nada**: Entorno de ejecución → Cambiar tipo de entorno
de ejecución → Acelerador por hardware → **GPU** (T4 gratis alcanza).

## 1. Montar Google Drive

Sube antes `chunks.jsonl` (o `chunks.jsonl.gz`) a una carpeta en tu Drive,
por ejemplo `MyDrive/codefest/chunks.jsonl`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ajusta estas rutas segun donde hayas subido el archivo en tu Drive
DRIVE_DIR = '/content/drive/MyDrive/codefest'
CHUNKS_PATH = f'{DRIVE_DIR}/chunks.jsonl'           # o chunks.jsonl.gz, ver celda siguiente
OUT_DIR = f'{DRIVE_DIR}/entrega_bge_m3'

import os
os.makedirs(OUT_DIR, exist_ok=True)
print('CHUNKS_PATH existe?', os.path.exists(CHUNKS_PATH))

In [ ]:
# Si subiste el .gz en vez del .jsonl sin comprimir, descomprime aqui:
import gzip, shutil, os

gz_path = CHUNKS_PATH + '.gz'
if not os.path.exists(CHUNKS_PATH) and os.path.exists(gz_path):
    with gzip.open(gz_path, 'rb') as f_in, open(CHUNKS_PATH, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print('Descomprimido a', CHUNKS_PATH)

with open(CHUNKS_PATH, encoding='utf-8') as f:
    n_lineas = sum(1 for _ in f)
print('lineas en chunks.jsonl:', n_lineas, '(debe ser 163625)')

## 2. Instalar dependencias

In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ADVERTENCIA: sin GPU activa. Ve a Entorno de ejecucion -> Cambiar tipo -> GPU y reinicia.')

## 3. Encoder (BAAI/bge-m3)

Misma lógica que `src/indexing/encoder.py` del repo, copiada aquí para que
el notebook sea autocontenido.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-m3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cargando {MODEL_NAME} en device={DEVICE}...')
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

def encode_texts(textos, batch_size=64):
    if not textos:
        return np.empty((0, 1024), dtype='float32')
    emb = model.encode(
        textos, batch_size=batch_size, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    )
    return emb.astype('float32')

## 4. Construir el índice FAISS

Misma lógica que `src/indexing/build_index.py`: lee `chunks.jsonl` en
superlotes, codifica, arma un `IndexFlatIP` y guarda `metadata.jsonl` en
el mismo orden de inserción (id interno FAISS = número de línea).

In [ ]:
import json
import faiss

SUPERBATCH_SIZE = 4000   # con GPU se puede subir bastante mas que en CPU
BATCH_SIZE = 64

def leer_chunks(path, limit=None):
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if line:
                yield json.loads(line)

def construir_indice(chunks_path, out_dir, limit=None):
    index = None
    registros = []
    superbatch = []

    def flush(lote):
        nonlocal index
        if not lote:
            return
        textos = [r['texto'] for r in lote]
        emb = encode_texts(textos, batch_size=BATCH_SIZE)
        if index is None:
            index = faiss.IndexFlatIP(emb.shape[1])
        index.add(emb)
        registros.extend(lote)

    for registro in leer_chunks(chunks_path, limit=limit):
        superbatch.append(registro)
        if len(superbatch) >= SUPERBATCH_SIZE:
            flush(superbatch)
            print(f'  ... {len(registros)} chunks codificados')
            superbatch = []
    flush(superbatch)

    os.makedirs(out_dir, exist_ok=True)
    faiss.write_index(index, os.path.join(out_dir, 'index.faiss'))
    with open(os.path.join(out_dir, 'metadata.jsonl'), 'w', encoding='utf-8') as f:
        for r in registros:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

    print(f'Indice FAISS con {index.ntotal} vectores guardado en {out_dir}')
    return index

### 4a. Prueba rápida primero (opcional pero recomendado)

Corre esto con `limit=50` para confirmar que todo funciona antes de lanzar
el corpus completo (163,625 chunks puede tardar un rato incluso con GPU).

In [ ]:
_ = construir_indice(CHUNKS_PATH, '/content/smoke_test', limit=50)

### 4b. Corpus completo

In [ ]:
construir_indice(CHUNKS_PATH, OUT_DIR)

## 5. Verificar el resultado

In [ ]:
idx = faiss.read_index(os.path.join(OUT_DIR, 'index.faiss'))
print('vectores en el indice:', idx.ntotal, '(debe ser 163625)')
with open(os.path.join(OUT_DIR, 'metadata.jsonl'), encoding='utf-8') as f:
    n = sum(1 for _ in f)
print('lineas en metadata.jsonl:', n, '(debe ser 163625)')

## 6. Listo

El resultado ya quedó guardado directamente en tu Google Drive, en
`MyDrive/codefest/entrega_bge_m3/` (`index.faiss` + `metadata.jsonl`).
Descarga esa carpeta y ubícala en el repo local en
`entrega/base_vectorial/encoder_bge_m3/`, o compártela directamente.